# Capstone Project: Slogan Classifier and Generator

In this capstone project you will train a Long Short-Term Memory (LSTM) model to generate slogans for businesses based on their industry, and also train a classifier to predict the industry based on a given slogan.

##Libraries
We recommend running this notebook using [Google Colab](https://colab.google/) however if you choose to use your local machine you will need to install spaCy before starting.

To install spaCy, refer to the installation instructions provided on the spaCy [website](https://spacy.io/usage). Note you may need to install an older version of Python that is compatible with spaCy. You can create a virtual environment for this project to install the specific version of Python that you need.

In [2]:
%pip install tensorflow


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Had to install TensorFlow and run Python 3.12.10 in order for everything to run properly on VS Code.


In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam
import spacy # available on Google Colab
from sklearn.model_selection import train_test_split

## Loading and viewing the dataset

- Load the slogan dataset into a variable called data.
- Extract relevant columns in a variable called df.
- Handle missing values.

Do **not** change the column names.

If you are using Google Colab you will need mount your Google Drive as follows:  
`from google.colab import drive`  
`drive.mount('/content/drive')`  

The path you use when loading your data will look something like this if you are using your Google Drive:  
"/content/drive/MyDrive/Colab Notebooks/slogan-valid.csv"

In [4]:
# Load the full slogan dataset into a variable called 'data'
data = pd.read_csv("slogan-valid.csv")

# Extract only the relevant columns: 'output' (the slogan) and 'industry'
df = data[["output", "industry"]].copy()

# Remove rows with missing values to ensure clean data
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

df.head()


,output,industry
0,Taking Care of Small Business Technology,computer hardware
1,Build World-Class Recreation Programs,"health, wellness and fitness"
2,Most Powerful Lead Generation Software for Mar...,internet
3,Hire quality freelancers for your job,internet
4,"Financial Advisers Norwich, Norfolk",financial services


## Data Preprocessing

Since we are working with textual data, we need software that understands natural language. For this, we'll use a library for processing text called **spaCy**. Using spaCy, we'll break the text into smaller units called tokens that are easier for the machine to process. This process is called **tokenisation**. We'll also convert all text to lowercase and remove punctuation because this information is not necessary for our models. Run the code below, and your dataframe (df) will gain a new column called **'processed_slogan'** which contains the preprocessed text.




In [5]:
# Load spaCy model for text processing
nlp = spacy.load("en_core_web_sm")

# Define text preprocessing function
def preprocess_text(text):
    text_lower = text.lower()
    doc = nlp(text_lower)

    processed_tokens = []

    for token in doc:
        if not token.is_punct:
            processed_tokens.append(token.text)

    return " ".join(processed_tokens)

df["processed_slogan"] = df["output"].apply(preprocess_text)

df.head()

,output,industry,processed_slogan
0,Taking Care of Small Business Technology,computer hardware,taking care of small business technology
1,Build World-Class Recreation Programs,"health, wellness and fitness",build world class recreation programs
2,Most Powerful Lead Generation Software for Mar...,internet,most powerful lead generation software for mar...
3,Hire quality freelancers for your job,internet,hire quality freelancers for your job
4,"Financial Advisers Norwich, Norfolk",financial services,financial advisers norwich norfolk


We want our model to generate **industry-specific** slogans. If we use the 'processed_slogan' column as it is, we'll be leaving out crucial context - the industries of the companies behind those slogans. To fix this, we'll create a new **'modified_slogan'** column that adds the industry name to the front of processed slogan.  

For example:  

> industry = 'computer hardware'  
processed_slogan = 'taking care of small business technology'  
modified_slogan = 'computer hardware taking care of small business technology'

Write code in the cell below to achieve this.

In [ ]:
#Prepend the industry name to each processed slogan to give the model industry context for example industry='internet', processed_slogan='connecting the world' => 'internet connecting the world'
df["modified_slogan"] = df["industry"] + " " + df["processed_slogan"]

df.head()


,output,industry,processed_slogan,modified_slogan
0,Taking Care of Small Business Technology,computer hardware,taking care of small business technology,computer hardware taking care of small busines...
1,Build World-Class Recreation Programs,"health, wellness and fitness",build world class recreation programs,"health, wellness and fitness build world class..."
2,Most Powerful Lead Generation Software for Mar...,internet,most powerful lead generation software for mar...,internet most powerful lead generation softwar...
3,Hire quality freelancers for your job,internet,hire quality freelancers for your job,internet hire quality freelancers for your job
4,"Financial Advisers Norwich, Norfolk",financial services,financial advisers norwich norfolk,financial services financial advisers norwich ...


Now we need to get data to train our model. We have textual data which we will need to represent numerically for our model to learn from it.  
The code below does the following:
1. Tokenizes a dataset of slogans.
2. Converts words to numerical indices.
3. Creates input sequences using the numerical indices.  

Here's how it works. From the 'modified_slogan' column, we take the slogan "computer hardware taking care of small business technology". The tokenisation process will convert words into their corresponding indices:  

<center>

| Word         | Token Index |
|-------------|-------|
| "computer"  | 1     |
| "hardware"  | 2     |
| "taking"    | 3     |
| "care"      | 4     |
| "of"        | 5     |
| "small"     | 6     |
| "business"  | 7     |
| "technology"| 8     |

</center>

So the tokenized list is:

<center>
[1, 2, 3, 4, 5, 6, 7, 8]
</center>

When creating input sequences for training, the loop generates progressively longer sequences.

<center>

| Token Index Sequence               | Corresponding Slogan                                 |
|------------------------------|-----------------------------------------------------|
| [1, 2]                       | "computer hardware"                                |
| [1, 2, 3]                    | "computer hardware taking"                        |
| [1, 2, 3, 4]                 | "computer hardware taking care"                   |
| [1, 2, 3, 4, 5]              | "computer hardware taking care of"                |
| [1, 2, 3, 4, 5, 6]           | "computer hardware taking care of small"          |
| [1, 2, 3, 4, 5, 6, 7]        | "computer hardware taking care of small business" |
| [1, 2, 3, 4, 5, 6, 7, 8]     | "computer hardware taking care of small business technology" |

</center>

Instead of training the model on only **complete slogans**, we provide partial phrases which will help the model learn how words connect over time. This will make it better at predicting the next word when generating slogans.  

Run the cell block below to generate the input sequences. Be sure to read the comments to understand what the code is doing.


In [7]:
'''** Clean up comments'''

# Tokenizer to convert words into numerical values tokens
tokenizer = Tokenizer()

# Tokenizer learns words in dataset
tokenizer.fit_on_texts(df["modified_slogan"])

# Total number of unique words in learned vocabulary
total_words = len(tokenizer.word_index) + 1

# Dictionary mapping words to its numerical index: index based on frequency i.e., more freq => lower index
tokenizer.word_index

# Creating input sequences
# Initialise list to store the input sequences
input_sequences = []

# Iterate over processed slogans
for line in df["modified_slogan"]:

    # Convert slogans to token sequences
    token_list = tokenizer.texts_to_sequences([line])[0] # returns list containing list of words indices; extracting inner list [0]

    # token_list is a list of tokenized word INDICES
    # Building list of progressively longer input sequences for better training
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

The input sequences created above are of **varying lengths**, which will be a problem when training our LSTM model. LSTMs require input sequences of **equal length**. So, we need to **pad** shorter sequences by **prepending zeros** until they match the length of the longest sequence.  

For example, if the longest sequence has **10 tokens**, our padded sequences will look like this:

<center>

| Input Sequence                     | Padded Sequence                         |
|-------------------------------------|-----------------------------------------|
| [1, 2]                              | [0, 0, 0, 0, 0, 0, 0, 0, 1, 2]         |
| [1, 2, 3]                           | [0, 0, 0, 0, 0, 0, 0, 1, 2, 3]         |
| [1, 2, 3, 4]                        | [0, 0, 0, 0, 0, 0, 1, 2, 3, 4]         |
| [1, 2, 3, 4, 5]                     | [0, 0, 0, 0, 0, 1, 2, 3, 4, 5]         |
| [1, 2, 3, 4, 5, 6]                  | [0, 0, 0, 0, 1, 2, 3, 4, 5, 6]         |
| [1, 2, 3, 4, 5, 6, 7]               | [0, 0, 0, 1, 2, 3, 4, 5, 6, 7]         |
| [1, 2, 3, 4, 5, 6, 7, 8]            | [0, 0, 1, 2, 3, 4, 5, 6, 7, 8]         |

</center>

In the cell below, write code that **finds the length of the longest sequence** in **input_sequences** and stores this value in a variable named **max_seq_len**.


In [ ]:
#Find the length of the longest sequence so all sequences can be padded to the same length
max_seq_len = max(len(seq) for seq in input_sequences)

print(f"Maximum sequence length: {max_seq_len}")


Maximum sequence length: 15


Run the cell below to pad the input sequences so they are all the same length as **max_seq_length**.

In [9]:
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding="pre")

## Training Data for Slogan Generator

The input sequences generated will be used as our training data. Our LSTM needs to learn how to predict the **next word** in a sequence.  

The inputs for our model will be the input sequences **excluding the last token index** and the outputs will be the **last token index**.  

As an example, let us use the input sequence [0, 0, 1, 2, 3, 4, 5, 6, 7, 8] and say it corresponds to the slogan "computer hardware taking care of small business technology". When training the model:

> Our input **x** will be the input sequence [0, 0, 1, 2, 3, 4, 5, 6, 7] corresponding to "computer hardware taking care of small".  
> Our output **y** will be [8] which corresponds to "business".  

In the code cell below, use `input_sequences` to create the following two variables:
1. **X_gen** which contains the input sequences excluding the last token index.
2. **y_gen** which contains the last token index of the input sequence.

In [ ]:
#X_gen: all tokens except the last (the context the model learns from)
#y_gen: only the last token (the word the model must predict)
X_gen = input_sequences[:, :-1]
y_gen = input_sequences[:, -1]

print(f"X_gen shape: {X_gen.shape}")
print(f"y_gen shape: {y_gen.shape}")


X_gen shape: (34736, 14)
y_gen shape: (34736,)


The model will output the next word of a sequence over a probability distribution. We need to encode our output variable for this to be possible.

In the code cell below, write code that will apply one-hot encoding to **y_gen** using `tf.keras.utils.to_categorical()`. **Maintain the same variable name**.  

*Hint: set the `num_classes` (number of classes) parameter to the total number of unique words in the learned vocabulary. You can access this value through a variable that was created when generating input sequences earlier.*

In [ ]:
#One-hot encode y_gen so the model outputs a probability over the full vocabulary
y_gen = tf.keras.utils.to_categorical(y_gen, num_classes=total_words)

print(f"y_gen shape after one-hot encoding: {y_gen.shape}")


y_gen shape after one-hot encoding: (34736, 6046)


## Slogan Generator Architecture

In the code cell that follows, configure the LSTM following these steps:

1. Create a sequential model using `tf.keras.models.Sequential()`. This model will have an embedding layer, two LSTM layers, and a dense output layer.
2. Add an embedding layer that converts words into dense vector representations. This layer should:
> *   Have `total_words`as the vocabulary size.
> *   Use 100 as an embedding dimension.
> *   Takes an input length of `max_seq_len - 1` (excludes the target word).
3. Add two LSTM layers.
> *   The first LSTM layer should have 150 **and** set `return_sequences` to `True`.
> *   The second LSTM layer should have 100 units.
4. Add a dense output layer which:
> *   Uses `total_words` as the number of units (one for each word in the vocabulary).
> *   Uses a softmax activation function.
5. Use `Sequential` to put everything together in the correct order to complete the architecture of the LSTM model called **gen_model**.


In [ ]:
#Build the LSTM slogan generator with an embedding layer, two LSTM layers and a dense output layer that predicts the next word over the vocabulary
gen_model = tf.keras.models.Sequential([
    #Embedding layer: maps word indices to dense vectors of size 100
    Embedding(total_words, 100, input_length=max_seq_len - 1),

    #First LSTM layer: 150 units; return_sequences=True feeds sequences to the next LSTM
    LSTM(150, return_sequences=True),

    #Second LSTM layer: 100 units; produces a single context vector
    LSTM(100),

    #Dense output layer: one unit per vocabulary word, softmax for probability distribution
    Dense(total_words, activation="softmax")
])

gen_model.summary()


c:\Users\bigma\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In the code cell below, compile `gen_model` using `categorical_crossentropy` loss, an Adam optimiser, and an appropriate metric of your choice.


In [ ]:
#Compile the generator model categorical_crossentropy is appropriate for multi-class classification with one-hot labels
gen_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(),
    metrics=["accuracy"]
)


## Slogan Generation

In the code cell below, fit the compiled model on the inputs and outputs, setting the **number of epochs to 50**.

In [ ]:
#Train the generator model for 50 epochs on the padded input sequences
gen_model.fit(X_gen, y_gen, epochs=50, verbose=1)


Epoch 1/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 19s 15ms/step - accuracy: 0.0709 - loss: 7.0352
Epoch 2/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.1132 - loss: 6.3393
Epoch 3/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.1530 - loss: 5.9967
Epoch 4/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.1775 - loss: 5.7366
Epoch 5/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.1975 - loss: 5.4998
Epoch 6/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 0.2146 - loss: 5.2684
Epoch 7/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.2294 - loss: 5.0503
Epoch 8/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.2406 - loss: 4.8505
Epoch 9/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.2497 - loss: 4.6514
Epoch 10/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.2598 - loss: 4.4533
Epoch 11/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.2692 - loss: 4.2649
Epoch 12

Epoch 1/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 19s 15ms/step - accuracy: 0.0709 - loss: 7.0352
Epoch 2/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.1132 - loss: 6.3393
Epoch 3/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.1530 - loss: 5.9967
Epoch 4/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.1775 - loss: 5.7366
Epoch 5/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.1975 - loss: 5.4998
Epoch 6/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 0.2146 - loss: 5.2684
Epoch 7/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.2294 - loss: 5.0503
Epoch 8/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.2406 - loss: 4.8505
Epoch 9/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.2497 - loss: 4.6514
Epoch 10/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.2598 - loss: 4.4533
Epoch 11/50
1086/1086 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.2692 - loss: 4.2649
Epoch 12

We will now define a function called `generate_slogan` which will generate a slogan by predicting one word at a time based on a given starting phrase (the `seed_text`). This function will do this using our trained model, `gen_model`.

Here is a breakdown of how the algorithm works:  

Let us assume the dictionary mapping words to unique indices, `tokenizer.word_index`, looks like this:

> `{'computer': 1, 'hardware': 2, 'taking': 3, 'care': 4, 'of': 5}`

If the model's predicted index for the next word is 3 (`predicted_index = 3`), the loop will:

> Check 'computer' (index 1) → No match  
> Check 'hardware' (index 2) → No match  
> Check 'taking' (index 3) → Match found!  
> Assign output_word = "taking" and exit the loop.  

The `output_word` will be appended to the `seed_text`, and the process will continue to add words to the `seed_text` until we have reached the maximum number of words **or** an invalid prediction occurs.  

Carefully follow the code below and complete the missing parts as guided by the comments.

In [15]:
def generate_slogan(seed_text, max_words=20):
    for _ in range(max_words):

        # Tokenise and pad seed_text to match the expected input length
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding="pre")

        # Predict the probability distribution of the next word over the vocabulary
        predictions = gen_model.predict(token_list, verbose=0)

        # Identify the word index with the highest predicted probability
        predicted_index = np.argmax(predictions[0])

        output_word = None

        # Find the word that corresponds to the predicted index
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        # Stop generating if no valid word is found
        if output_word is None:
            break

        # Append the predicted word to the seed text and continue
        seed_text += " " + output_word

    return seed_text


## Training Data for Slogan Classifier

We will now prepare the data we will use to train our classifier. For our classifier, the inputs will come from the `processed_slogans` column of our DataFrame, `df`. The outputs will be the different industry categories under the `industry` column.

In the code cell below, extract the unique values from the `industry` column in the DataFrame and store these in a variable called **industries**.

In [ ]:
#Extract the unique industry categories from the dataset
industries = df["industry"].unique()

print(f"Number of unique industries: {len(industries)}")
print(industries)


Number of unique industries: 142
<StringArray>
[                   'computer hardware',
         'health, wellness and fitness',
                             'internet',
                   'financial services',
 'mechanical or industrial engineering',
            'marketing and advertising',
               'hospital & health care',
                             'research',
  'information technology and services',
                    'computer software',
 ...
              'paper & forest products',
                 'alternative medicine',
                        'public policy',
           'glass, ceramics & concrete',
                            'judiciary',
                            'libraries',
                         'fund-raising',
               'political organization',
             'package/freight delivery',
                         'supermarkets']
Length: 142, dtype: str


Create a dictionary called `industry_to_index` where each unique industry is mapped to a unique index starting from 0.

*Hint: Use the `enumerate()` function.*

In [ ]:
#Map each unique industry name to a unique integer index starting at 0
industry_to_index = {industry: idx for idx, industry in enumerate(industries)}

print(industry_to_index)


{'computer hardware': 0, 'health, wellness and fitness': 1, 'internet': 2, 'financial services': 3, 'mechanical or industrial engineering': 4, 'marketing and advertising': 5, 'hospital & health care': 6, 'research': 7, 'information technology and services': 8, 'computer software': 9, 'oil & energy': 10, 'dairy': 11, 'transportation/trucking/railroad': 12, 'design': 13, 'furniture': 14, 'professional training & coaching': 15, 'hospitality': 16, 'textiles': 17, 'food & beverages': 18, 'management consulting': 19, 'medical practice': 20, 'accounting': 21, 'performing arts': 22, 'electrical/electronic manufacturing': 23, 'higher education': 24, 'outsourcing/offshoring': 25, 'venture capital & private equity': 26, 'writing and editing': 27, 'mining & metals': 28, 'construction': 29, 'consumer electronics': 30, 'retail': 31, 'human resources': 32, 'staffing and recruiting': 33, 'farming': 34, 'wholesale': 35, 'events services': 36, 'import and export': 37, 'non-profit organization management

Create a new column `industry_index` in your DataFrame by mapping the `industry` column to the indices using the `industry_to_index` dictionary.

*Hint: Use the  `map()` function.*

In [ ]:
#Add a numerical industry_index column by mapping industry names to their assigned indices
df["industry_index"] = df["industry"].map(industry_to_index)

df.head()


,output,industry,processed_slogan,modified_slogan,industry_index
0,Taking Care of Small Business Technology,computer hardware,taking care of small business technology,computer hardware taking care of small busines...,0
1,Build World-Class Recreation Programs,"health, wellness and fitness",build world class recreation programs,"health, wellness and fitness build world class...",1
2,Most Powerful Lead Generation Software for Mar...,internet,most powerful lead generation software for mar...,internet most powerful lead generation softwar...,2
3,Hire quality freelancers for your job,internet,hire quality freelancers for your job,internet hire quality freelancers for your job,2
4,"Financial Advisers Norwich, Norfolk",financial services,financial advisers norwich norfolk,financial services financial advisers norwich ...,3


Split the DataFrame `df` into training and testing sets, setting aside 20% of the data for the test set. Be sure to set the parameter `stratify=df["industry_index"]`. This ensures that both sets have the same proportion of each class (industry) as in the original dataset, resulting in balanced datasets. Call the training DataFrame `df_train` and the testing DataFrame `df_test`.

In [ ]:
#Remove industries with only 1 sample — stratified splitting requires at least 2 per class
industry_counts = df["industry_index"].value_counts()
valid_industries = industry_counts[industry_counts >= 2].index
df = df[df["industry_index"].isin(valid_industries)].reset_index(drop=True)

#Split into 80% training and 20% testing sets stratify ensures each industry is proportionally represented in both splits
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["industry_index"]
)

print(f"Training set size: {len(df_train)}")
print(f"Testing set size:  {len(df_test)}")


Training set size: 4272
Testing set size:  1068


Our classifier will use padded slogan sequences as inputs, similar to input sequences used for the slogan generator. The difference is we will not use sequences that get progressively longer, but instead we will use **complete slogans**. This is because our classifier does not need to learn how to predict what word comes next. It needs the full context of a slogan to learn how to accurately predict the industry.  

The next steps will walk you through how to create these sequences.  

We previously created and fitted a `Tokenizer` object called `tokenizer` while preparing data for the slogan generator. Now, we will reuse it to convert words into numerical indices.  

In the code cell below, use the `texts_to_sequences()` **method** of `tokenizer` to transform the `processed_slogan` column in **both** the `df_train` and `df_test` DataFrames into sequences of numerical indices. Store the results in variables named `X_train` and `X_test`.


In [ ]:
#Convert the processed slogans to sequences of numerical word indices using the same tokenizer fitted during generator preprocessing
X_train = tokenizer.texts_to_sequences(df_train["processed_slogan"])
X_test = tokenizer.texts_to_sequences(df_test["processed_slogan"])


The slogan sequences are of varying lengths. We will need to pad them the same way we did to the input sequences for the slogan generator. The `pad_sequences()` function can ensure the sequences in `slogan_sequences` have the same length.  

In the code cell below, use the `pad_sequences()` function to standardise the `slogan_sequences` lengths. Set the `maxlen` parameter to `max_seq_len`, the `padding` parameter to 0, and assign the resulting padded sequences to the same variables, `X_train` and `X_test`.

In [ ]:
#Pad sequences with zeros at the front so all inputs are the same length (max_seq_len)
X_train = pad_sequences(X_train, maxlen=max_seq_len, padding="pre")
X_test = pad_sequences(X_test, maxlen=max_seq_len, padding="pre")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")


X_train shape: (4272, 15)
X_test shape:  (1068, 15)


We have successfully created training and testing inputs for our model. Now, we will create the outputs - industry categories.

 In the code cell that follows, use `tf.keras.utils.to_categorical()` to apply one-hot encoding to the `industry_index` column of **both** `df_train` and `df_test` DataFrames. Assign the results to a variables named `y_train` and `y_test`.

 *Hint: set the `num_classes` parameter to the total number of industries in the DataFrame. The `industries` variable can be used to find this value.*

In [ ]:
#One-hot encode the industry indices so the output matches the softmax classifier's format
y_train = tf.keras.utils.to_categorical(
    df_train["industry_index"], num_classes=len(industries)
)
y_test = tf.keras.utils.to_categorical(
    df_test["industry_index"], num_classes=len(industries)
)

print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")


y_train shape: (4272, 142)
y_test shape:  (1068, 142)


## Slogan Classifier Architecture

Configure the LSTM classifier following these steps:  


1. Create a Sequential model:  
   Use `tf.keras.models.Sequential()` to create a sequential model. This model will consist of an embedding layer, two LSTM layers, and a dense output layer.

2. Add an embedding layer which will convert words into dense vector representations. Configure this layer with:
   > * `total_words` as the vocabulary size.
   > * 100 as the embedding dimension.
   > * `max_seq_len` as the `input_length` (this is the length of the slogans).

3. Add the first LSTM layer. Configure it with:
   > * 150 units.
   > * Set `return_sequences` to `True` to ensure the layer outputs sequences for the next LSTM layer.

4. Add the second LSTM layer which will process the output from the previous LSTM layer. Configure it with:
   > * 100 units.
   > * No need to set `return_sequences` here (it is the final LSTM layer).

5. Add the dense output layer which will classify the data into industries. Configure it with:
   > * The number of unique industries as the number of units.
   > * The `softmax` activation function to get probabilities for each class (industry).

6. Use `Sequential` to arrange all layers in the correct order and complete the architecture of the LSTM model called **class_model**.


In [ ]:
# Build the LSTM slogan classifier: same architecture as the generator but the output layer predicts over industries rather than vocabulary words
class_model = tf.keras.models.Sequential([
    #Embedding layer: maps word indices to dense vectors of size 100
    Embedding(total_words, 100, input_length=max_seq_len),

    #First LSTM layer: 150 units; return_sequences=True feeds sequences to the next LSTM
    LSTM(150, return_sequences=True),

    #Second LSTM layer: 100 units; produces a single context vector
    LSTM(100),

    #Dense output layer: one unit per industry, softmax for probability distribution
    Dense(len(industries), activation="softmax")
])

class_model.summary()


c:\Users\bigma\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


c:\Users\bigma\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In the code cell below, compile `class_model` using `categorical_crossentropy` loss, an Adam optimiser, and an appropriate metric of your choice.

In [ ]:
#Compile the classifier model
class_model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(),
    metrics=["accuracy"]
)


## Slogan Classification & Evaluation

In the code cell that follows, fit the compiled model on the inputs and outputs, setting **the number of epochs to 50**.

In [ ]:
#Train the classifier model for 50 epochs, monitoring performance on the test set each epoch
class_model.fit(
    X_train,
    y_train,
    epochs=50,
    validation_data=(X_test, y_test),
    verbose=1
)


Epoch 1/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.0826 - loss: 4.3886 - val_accuracy: 0.0843 - val_loss: 4.2842
Epoch 2/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0847 - loss: 4.2867 - val_accuracy: 0.0843 - val_loss: 4.2714
Epoch 3/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0847 - loss: 4.2526 - val_accuracy: 0.0890 - val_loss: 4.1639
Epoch 4/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.1217 - loss: 3.9584 - val_accuracy: 0.1358 - val_loss: 3.9401
Epoch 5/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.2083 - loss: 3.4810 - val_accuracy: 0.1648 - val_loss: 3.7984
Epoch 6/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.2931 - loss: 3.0041 - val_accuracy: 0.1732 - val_loss: 3.8499
Epoch 7/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.3827 - loss: 2.6360 - val_accuracy: 0.1919 - val_loss: 3.9108
Epoch 8/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.4654 - loss: 2.3023 - val_accu

Epoch 1/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.0826 - loss: 4.3886 - val_accuracy: 0.0843 - val_loss: 4.2842
Epoch 2/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0847 - loss: 4.2867 - val_accuracy: 0.0843 - val_loss: 4.2714
Epoch 3/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0847 - loss: 4.2526 - val_accuracy: 0.0890 - val_loss: 4.1639
Epoch 4/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.1217 - loss: 3.9584 - val_accuracy: 0.1358 - val_loss: 3.9401
Epoch 5/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.2083 - loss: 3.4810 - val_accuracy: 0.1648 - val_loss: 3.7984
Epoch 6/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.2931 - loss: 3.0041 - val_accuracy: 0.1732 - val_loss: 3.8499
Epoch 7/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.3827 - loss: 2.6360 - val_accuracy: 0.1919 - val_loss: 3.9108
Epoch 8/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.4654 - loss: 2.3023 - val_accu

Evaluate the model using the testing set. Add a comment on the model's performance.

In [ ]:
#Evaluate the classifier on the held-out test set to measure generalisation performance
test_loss, test_accuracy = class_model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

#A higher accuracy indicates the model has successfully learned associations between slogan language patterns and industry categories.


Test Loss:     7.4056
Test Accuracy: 0.1966


We will now define a function called `classify_slogan` which takes a slogan as input and predicts the industry it belongs to using the trained model, `class_model`.  

Carefully follow the code below and complete the missing parts (indicated by ellipses) as guided by the comments.

In [29]:
def classify_slogan(slogan):
    # Clean the input slogan using the same preprocessing applied to training data
    slogan = preprocess_text(slogan)

    # Convert the slogan to a sequence of numerical word indices
    sequence = tokenizer.texts_to_sequences([slogan])

    # Pad the sequence to match the expected input length of the classifier
    padded_sequence = pad_sequences(sequence, maxlen=max_seq_len, padding="pre")

    # Pass the padded sequence through the classifier to get industry probabilities
    prediction = class_model.predict(padded_sequence, verbose=0)

    # Extract the index of the industry with the highest predicted probability
    predicted_index = np.argmax(prediction[0])

    # Return the industry name that corresponds to the predicted index
    return industries[predicted_index]


## Combining the two models

Run the code cell below to combine the two models: we will first generate a slogan for a company in the "internet" industry, then pass the generated slogan to the slogan classifier to see if it correctly classifies it as internet.

In [30]:
industry = "internet"
generated_slogan = generate_slogan(industry)
predicted_industry = classify_slogan(generated_slogan)

print(f"Generated Slogan: {generated_slogan}")
print(f"Predicted Industry: {predicted_industry}")

Generated Slogan: internet web design glasgow remarkable software for performance and wholesale investing alternative games supplier sale banking sale less cpr games companies
Predicted Industry: publishing


Compare the results and comment on any differences you notice between the generated slogans and the classifier’s predictions in the markdown cell below.


## Comparison of Generator and Classifier Results

The generator produced a slogan starting with "internet" but mixed in vocabulary from unrelated industries, resulting in a generic output. The classifier then predicted "publishing" instead of "internet".

This mismatch occurs because the generator learns to produce fluent word sequences rather than industry-specific ones. When the generated slogan lacks distinctive internet-related words, the classifier associates it with a different industry. Both models are also limited by the small amount of training data per industry.
